# From Hand-Derived Gradients to PyTorch Autograd

## The Exam Score Example — Three Ways

We'll solve the **same problem** three ways to see the evolution:

1. **100% by hand** — compute every gradient manually (Day 2 style)
2. **PyTorch autograd** — let PyTorch compute gradients (Day 3 style)
3. **PyTorch optimizer** — let PyTorch handle everything (real-world style)

### The Problem

Predict a student's exam score from:
- **h** = hours studied
- **s** = hours slept

Model: `score = w1 × hours_studied + w2 × hours_slept + b`

We need to learn **w1**, **w2**, and **b** from data.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

# Our data: 5 students
hours_studied = np.array([1.0, 2.0, 3.0, 5.0, 4.0])
hours_slept   = np.array([4.0, 7.0, 8.0, 6.0, 9.0])
actual_scores = np.array([40.0, 65.0, 75.0, 80.0, 90.0])

print("Our training data:")
print(f"{'Student':>8} | {'Studied (h)':>11} | {'Slept (s)':>9} | {'Score':>6}")
print("-" * 48)
for i in range(5):
    print(f"{'  #' + str(i+1):>8} | {hours_studied[i]:>11.0f} | {hours_slept[i]:>9.0f} | {actual_scores[i]:>6.0f}")

---

## Way 1: 100% By Hand (NumPy only — Day 2 style)

We compute EVERYTHING manually:
- Forward pass (prediction)
- Loss
- Gradients (using the formulas we derived)
- Weight updates

This is what we did in the dry run. Now let's code it.

In [ ]:
# WAY 1: Everything by hand with NumPy

h = hours_studied
s = hours_slept
actual = actual_scores

# Random starting weights
w1 = 1.0
w2 = 1.0
b  = 0.0
lr = 0.001

losses_hand = []

print(f"Start: w1={w1:.3f}, w2={w2:.3f}, b={b:.3f}\n")

for epoch in range(200):
    
    # ---- FORWARD PASS ----
    # We write the model equation
    prediction = w1 * h + w2 * s + b
    
    # ---- LOSS ----
    # We write the loss formula
    error = prediction - actual
    loss = np.mean(error ** 2)
    losses_hand.append(loss)
    
    # ---- GRADIENTS (we derived these by hand!) ----
    # ∂loss/∂w1 = mean(2 × error × h)   ← h because w1 multiplies h
    # ∂loss/∂w2 = mean(2 × error × s)   ← s because w2 multiplies s  
    # ∂loss/∂b  = mean(2 × error × 1)   ← 1 because b is just added
    dw1 = np.mean(2 * error * h)
    dw2 = np.mean(2 * error * s)
    db  = np.mean(2 * error * 1)
    
    # ---- UPDATE ----
    # We manually update each weight
    w1 = w1 - lr * dw1
    w2 = w2 - lr * dw2
    b  = b  - lr * db
    
    if epoch % 40 == 0:
        print(f"Epoch {epoch:3d}: loss={loss:8.1f}, w1={w1:.3f}, w2={w2:.3f}, b={b:.3f}")

print(f"\nFinal: w1={w1:.3f}, w2={w2:.3f}, b={b:.3f}")
print(f"Reads: score = {w1:.1f} × study_hours + {w2:.1f} × sleep_hours + {b:.1f}")

### What we had to do manually:

```python
dw1 = np.mean(2 * error * h)    # ← WE derived this formula
dw2 = np.mean(2 * error * s)    # ← WE derived this formula
db  = np.mean(2 * error * 1)    # ← WE derived this formula

w1 = w1 - lr * dw1              # ← WE updated each weight individually
w2 = w2 - lr * dw2
b  = b  - lr * db
```

**6 lines of hand-derived math.** For 3 parameters, this is fine. For 3 million? No way.

Let's save these final weights so we can compare later:

---

## Way 2: PyTorch Autograd (Day 3 style)

Same problem, same data, same starting weights. But now **PyTorch computes the gradients for us**.

The ONLY change: we replace the 3 hand-derived gradient lines with `loss.backward()`.

In [ ]:
# WAY 2: PyTorch autograd — gradients computed automatically

# Same data, but as PyTorch tensors now
h_t = torch.tensor(hours_studied, dtype=torch.float32)
s_t = torch.tensor(hours_slept, dtype=torch.float32)
actual_t = torch.tensor(actual_scores, dtype=torch.float32)

# Same starting weights — but with requires_grad=True!
# This single flag tells PyTorch: "track everything that happens to these"
w1_t = torch.tensor(1.0, requires_grad=True)
w2_t = torch.tensor(1.0, requires_grad=True)
b_t  = torch.tensor(0.0, requires_grad=True)
lr = 0.001

losses_autograd = []

print(f"Start: w1={w1_t.item():.3f}, w2={w2_t.item():.3f}, b={b_t.item():.3f}\n")

for epoch in range(200):
    
    # ---- FORWARD PASS (identical to Way 1) ----
    prediction = w1_t * h_t + w2_t * s_t + b_t
    
    # ---- LOSS (identical to Way 1) ----
    loss = torch.mean((prediction - actual_t) ** 2)
    losses_autograd.append(loss.item())
    
    # ---- GRADIENTS (THE BIG CHANGE!) ----
    # Instead of 3 hand-derived formulas, just ONE line:
    loss.backward()
    
    # PyTorch has now filled in:
    #   w1_t.grad → same value as our hand-computed dw1
    #   w2_t.grad → same value as our hand-computed dw2
    #   b_t.grad  → same value as our hand-computed db
    
    # ---- UPDATE (same idea, but must use torch.no_grad) ----
    with torch.no_grad():
        w1_t -= lr * w1_t.grad
        w2_t -= lr * w2_t.grad
        b_t  -= lr * b_t.grad
    
    # ---- ZERO GRADIENTS (new! PyTorch accumulates by default) ----
    w1_t.grad.zero_()
    w2_t.grad.zero_()
    b_t.grad.zero_()
    
    if epoch % 40 == 0:
        print(f"Epoch {epoch:3d}: loss={loss.item():8.1f}, w1={w1_t.item():.3f}, w2={w2_t.item():.3f}, b={b_t.item():.3f}")

print(f"\nFinal: w1={w1_t.item():.3f}, w2={w2_t.item():.3f}, b={b_t.item():.3f}")

### Let's PROVE that PyTorch computes the same gradients as our hand formulas

This is the most important cell in this notebook. Let's run ONE step of both and compare the exact gradient values:

In [ ]:
# PROOF: hand gradients == PyTorch gradients

# Starting point: w1=1, w2=1, b=0 (same as both examples above)
h_np = hours_studied
s_np = hours_slept
actual_np = actual_scores

# ---- HAND COMPUTATION ----
pred_np = 1.0 * h_np + 1.0 * s_np + 0.0
error_np = pred_np - actual_np
hand_dw1 = np.mean(2 * error_np * h_np)
hand_dw2 = np.mean(2 * error_np * s_np)
hand_db  = np.mean(2 * error_np * 1)

# ---- PYTORCH COMPUTATION ----
w1_check = torch.tensor(1.0, requires_grad=True)
w2_check = torch.tensor(1.0, requires_grad=True)
b_check  = torch.tensor(0.0, requires_grad=True)

h_check = torch.tensor(hours_studied, dtype=torch.float32)
s_check = torch.tensor(hours_slept, dtype=torch.float32)
a_check = torch.tensor(actual_scores, dtype=torch.float32)

pred_t = w1_check * h_check + w2_check * s_check + b_check
loss_t = torch.mean((pred_t - a_check) ** 2)
loss_t.backward()

# ---- COMPARE ----
print("Gradient Comparison: Hand vs PyTorch")
print("=" * 55)
print(f"{'':>15} | {'Hand (NumPy)':>12} | {'PyTorch':>12} | {'Match?':>6}")
print("-" * 55)
print(f"{'∂loss/∂w1':>15} | {hand_dw1:>12.4f} | {w1_check.grad.item():>12.4f} | {'  ✓' if abs(hand_dw1 - w1_check.grad.item()) < 0.01 else '  ✗'}")
print(f"{'∂loss/∂w2':>15} | {hand_dw2:>12.4f} | {w2_check.grad.item():>12.4f} | {'  ✓' if abs(hand_dw2 - w2_check.grad.item()) < 0.01 else '  ✗'}")
print(f"{'∂loss/∂b':>15} | {hand_db:>12.4f} | {b_check.grad.item():>12.4f} | {'  ✓' if abs(hand_db - b_check.grad.item()) < 0.01 else '  ✗'}")
print()
print("EXACT SAME VALUES!")
print("loss.backward() computed the same thing as our hand-derived formulas.")
print("But it figured out the formulas AUTOMATICALLY from the forward pass.")

### This is the key moment:

```python
# HAND (Day 2) — you figure out 3 formulas:
dw1 = np.mean(2 * error * h)
dw2 = np.mean(2 * error * s)
db  = np.mean(2 * error)

# PYTORCH (Day 3) — one line, same result:
loss.backward()
```

Same numbers. Same result. But PyTorch didn't need us to derive anything.

---

## Way 3: PyTorch Optimizer (Real-World Style)

In Way 2, we still manually wrote `w1 -= lr * w1.grad` for each weight.

In real projects, an **optimizer** handles all updates for you. This is the pattern used in every PyTorch project from tutorials to GPT:

```python
optimizer.zero_grad()   # 1. clear old gradients
loss.backward()          # 2. compute new gradients
optimizer.step()         # 3. update ALL weights at once
```

Three lines. That's the whole training loop inside.

In [ ]:
# WAY 3: PyTorch with optimizer — the real-world pattern

h_t = torch.tensor(hours_studied, dtype=torch.float32)
s_t = torch.tensor(hours_slept, dtype=torch.float32)
actual_t = torch.tensor(actual_scores, dtype=torch.float32)

# Same starting weights
w1_o = torch.tensor(1.0, requires_grad=True)
w2_o = torch.tensor(1.0, requires_grad=True)
b_o  = torch.tensor(0.0, requires_grad=True)

# The optimizer manages ALL weights and their updates
# SGD = Stochastic Gradient Descent (does exactly: param -= lr * param.grad)
optimizer = torch.optim.SGD([w1_o, w2_o, b_o], lr=0.001)

losses_optimizer = []

print(f"Start: w1={w1_o.item():.3f}, w2={w2_o.item():.3f}, b={b_o.item():.3f}\n")

for epoch in range(200):
    
    # ---- FORWARD PASS (we only write this) ----
    prediction = w1_o * h_t + w2_o * s_t + b_o
    
    # ---- LOSS (we only write this) ----
    loss = torch.mean((prediction - actual_t) ** 2)
    losses_optimizer.append(loss.item())
    
    # ---- THE MAGIC 3 LINES (everything else is handled) ----
    optimizer.zero_grad()   # clear old gradients
    loss.backward()          # compute all gradients automatically
    optimizer.step()         # update all weights automatically
    
    if epoch % 40 == 0:
        print(f"Epoch {epoch:3d}: loss={loss.item():8.1f}, w1={w1_o.item():.3f}, w2={w2_o.item():.3f}, b={b_o.item():.3f}")

print(f"\nFinal: w1={w1_o.item():.3f}, w2={w2_o.item():.3f}, b={b_o.item():.3f}")

## Comparing All Three — Same Results, Less Code

Let's plot the loss curves and verify they're identical:

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: All 3 loss curves (should overlap perfectly)
ax1.plot(losses_hand, 'b-', linewidth=3, label='Way 1: Hand (NumPy)', alpha=0.7)
ax1.plot(losses_autograd, 'r--', linewidth=2, label='Way 2: PyTorch autograd')
ax1.plot(losses_optimizer, 'g:', linewidth=2, label='Way 3: PyTorch optimizer')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('All 3 Methods → Same Loss Curve')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right: Code comparison
methods = ['Way 1\nHand\n(NumPy)', 'Way 2\nAutograd\n(PyTorch)', 'Way 3\nOptimizer\n(PyTorch)']
gradient_lines = [3, 1, 1]   # lines of code for gradient computation
update_lines = [3, 3, 1]     # lines of code for weight updates
other_lines = [0, 3, 1]      # zero_grad, no_grad, etc.

x = np.arange(len(methods))
width = 0.25

bars1 = ax2.bar(x - width, gradient_lines, width, label='Gradient code', color='#e74c3c')
bars2 = ax2.bar(x, update_lines, width, label='Update code', color='#3498db')
bars3 = ax2.bar(x + width, other_lines, width, label='Bookkeeping', color='#95a5a6')

ax2.set_ylabel('Lines of code')
ax2.set_title('Code Needed for Each Method')
ax2.set_xticks(x)
ax2.set_xticklabels(methods)
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("All 3 produce IDENTICAL results.")
print("But look at how much less code Way 3 needs!")
print("And Way 3 scales to billions of parameters with zero changes.")

## Let's Test the Model — Does It Actually Work?

In [ ]:
# How well does our trained model predict?
w1_final = w1_o.item()
w2_final = w2_o.item()
b_final = b_o.item()

print(f"Learned model: score = {w1_final:.2f} × study_hours + {w2_final:.2f} × sleep_hours + {b_final:.2f}")
print(f"\nIn plain English:")
print(f"  Each hour of studying adds ~{w1_final:.0f} points")
print(f"  Each hour of sleep adds ~{w2_final:.0f} points")
print(f"  Base score (no study, no sleep): ~{b_final:.0f} points")

print(f"\n{'Student':>8} | {'Studied':>7} | {'Slept':>5} | {'Predicted':>9} | {'Actual':>6} | {'Error':>6}")
print("-" * 60)
for i in range(5):
    pred = w1_final * hours_studied[i] + w2_final * hours_slept[i] + b_final
    err = pred - actual_scores[i]
    print(f"{'  #'+str(i+1):>8} | {hours_studied[i]:>7.0f} | {hours_slept[i]:>5.0f} | {pred:>9.1f} | {actual_scores[i]:>6.0f} | {err:>+6.1f}")

# Predict for NEW students (never seen during training!)
print(f"\n--- Predictions for new students ---")
new_students = [
    (6, 8, "Studies hard, sleeps well"),
    (1, 3, "Barely studied, barely slept"),
    (3, 10, "Average study, great sleep"),
]
for study, sleep, desc in new_students:
    pred = w1_final * study + w2_final * sleep + b_final
    print(f"  {desc}: {study}h study, {sleep}h sleep → predicted score: {pred:.0f}")

---

## The Big Picture: Side-by-Side Code Comparison

Here's every line of the training loop for all 3 approaches:

```python
# ╔══════════════════════════════════════════════════════════════════╗
# ║  WAY 1: Hand (Day 2)          — 9 lines inside the loop        ║
# ╚══════════════════════════════════════════════════════════════════╝
prediction = w1 * h + w2 * s + b                 # forward pass
error = prediction - actual
loss = np.mean(error ** 2)                        # loss
dw1 = np.mean(2 * error * h)                      # gradient (HAND DERIVED)
dw2 = np.mean(2 * error * s)                      # gradient (HAND DERIVED)
db  = np.mean(2 * error * 1)                      # gradient (HAND DERIVED)
w1 = w1 - lr * dw1                                # update (MANUAL)
w2 = w2 - lr * dw2                                # update (MANUAL)
b  = b  - lr * db                                 # update (MANUAL)


# ╔══════════════════════════════════════════════════════════════════╗
# ║  WAY 2: Autograd (Day 3)      — 8 lines (no hand gradients!)   ║
# ╚══════════════════════════════════════════════════════════════════╝
prediction = w1 * h + w2 * s + b                 # forward pass
loss = torch.mean((prediction - actual) ** 2)     # loss
loss.backward()                                    # ALL gradients (AUTOMATIC!)
with torch.no_grad():
    w1 -= lr * w1.grad                             # update (still manual)
    w2 -= lr * w2.grad
    b  -= lr * b.grad
w1.grad.zero_(); w2.grad.zero_(); b.grad.zero_()  # clear gradients


# ╔══════════════════════════════════════════════════════════════════╗
# ║  WAY 3: Optimizer (Real-world) — 5 lines (clean & scalable!)   ║
# ╚══════════════════════════════════════════════════════════════════╝
prediction = w1 * h + w2 * s + b                 # forward pass
loss = torch.mean((prediction - actual) ** 2)     # loss
optimizer.zero_grad()                              # clear gradients
loss.backward()                                    # ALL gradients (AUTOMATIC!)
optimizer.step()                                   # ALL updates (AUTOMATIC!)
```

### The evolution:

| | Gradient computation | Weight update | Scales to GPT? |
|---|---|---|---|
| **Way 1** | 3 lines of formulas we derived | 3 lines, one per weight | No way |
| **Way 2** | 1 line: `loss.backward()` | 3 lines, one per weight | Gradients yes, updates no |
| **Way 3** | 1 line: `loss.backward()` | 1 line: `optimizer.step()` | **Yes!** Same 3 lines for ANY model |

---

## What You've Learned

1. **Day 2 gave you the understanding:** You know what `loss.backward()` computes inside (2 × error × input for each weight), and WHY (chain rule).

2. **Day 3 gave you the tool:** You'll never derive gradients by hand again. You write the forward pass, PyTorch handles the rest.

3. **The 3-line pattern** is the entire training loop for every model from linear regression to GPT:
```python
optimizer.zero_grad()
loss.backward()
optimizer.step()
```

From now on, all our energy goes into designing better **forward passes** (models). The backward pass is always free.